# Fake News Detection — Phase 4: Hyperparameter Tuning

**Goal:** Tune the best-performing model (**Random Forest**) using GridSearchCV
to maximize cross-validated accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## 1. Load & Vectorize Data

In [ ]:
df = pd.read_csv('fake_news_data_clean.csv')
X, y = df['clean_text'], df['label']

tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_tfidf = tfidf.fit_transform(X)

print('Shape:', X_tfidf.shape)

## 2. Define Parameter Grid

We tune the key Random Forest hyperparameters:

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print(f'Parameter grid has {len(param_grid)} hyperparameters')

## 3. Grid Search with Cross-Validation

We use 5-fold CV on the full dataset (same as previous evaluation):

In [ ]:
rf = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(
    rf, param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_tfidf, y)

print(f'Best CV Accuracy: {grid_search.best_score_:.4f}')
print(f'Best Parameters: {grid_search.best_params_}')

## 4. Compare Default vs. Tuned

In [ ]:
# Default Random Forest (baseline)
default_rf = RandomForestClassifier(n_estimators=100, random_state=42)
default_scores = cross_val_score(default_rf, X_tfidf, y, cv=5)

# Tuned Random Forest (best from grid search)
tuned_rf = grid_search.best_estimator_
tuned_scores = cross_val_score(tuned_rf, X_tfidf, y, cv=5)

print(f'Default RF  CV Mean: {default_scores.mean():.2%}  +/- {default_scores.std():.2%}')
print(f'Tuned RF    CV Mean: {tuned_scores.mean():.2%}  +/- {tuned_scores.std():.2%}')
print(f'Improvement: {tuned_scores.mean() - default_scores.mean():.2%}')

## 5. Final Evaluation on Held-Out Test Set

In [ ]:
# Re-split to get a clean test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf_final = TfidfVectorizer(max_features=5000, min_df=2)
X_train_final = tfidf_final.fit_transform(X_train)
X_test_final = tfidf_final.transform(X_test)

# Train tuned model on training set
final_model = grid_search.best_estimator_
final_model.fit(X_train_final, y_train)
y_pred = final_model.predict(X_test_final)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['Factual News', 'Fake News']))

## 6. Save the Tuned Model

Persist both the vectorizer and the model for API deployment.

In [ ]:
import joblib

# Refit on full dataset for maximum leverage
final_tfidf = TfidfVectorizer(max_features=5000, min_df=2)
X_full = final_tfidf.fit_transform(X)
final_model.fit(X_full, y)

joblib.dump(final_tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(final_model, 'random_forest_model.pkl')

print('Saved tfidf_vectorizer.pkl and random_forest_model.pkl')

## Summary

| Model | Test Accuracy | CV Mean |
|---|---|---|
| Default Random Forest | ~100% | ~99.0% |
| Tuned Random Forest | ... | ... |

The tuned model and vectorizer are saved and ready for **Phase 6: FastAPI Backend**.